# Imports & Load Artifacts

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load clustered customer data
df = pd.read_csv("processed/customer_rfm_clustered.csv")
# Load cluster profile
cluster_profile = pd.read_csv("models/cluster_profile.csv")
df.head()

,customer_unique_id,Recency,Frequency,Monetary,avg_delivery_time,avg_items,avg_installments,cluster
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,6.0,1.0,8.0,3
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,3.0,1.0,1.0,3
2,0000f46a3911fa3c0805444483337064,537,1,86.22,25.0,1.0,8.0,0
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,20.0,1.0,4.0,0
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,13.0,1.0,6.0,0


# Normalize RFM for Scoring (0–1 scale)

In [3]:
df['R_norm'] = 1 - (df['Recency'] - df['Recency'].min()) / (df['Recency'].max() - df['Recency'].min())
df['F_norm'] = (df['Frequency'] - df['Frequency'].min()) / (df['Frequency'].max() - df['Frequency'].min())
df['M_norm'] = (df['Monetary'] - df['Monetary'].min()) / (df['Monetary'].max() - df['Monetary'].min())

# Customer Experience Index

In [4]:
df['experience_index'] = (
    0.4 * df['R_norm'] +
    0.3 * df['F_norm'] +
    0.3 * df['M_norm']
).round(3)

# Experience Level Labels

In [5]:
def experience_label(score):
    if score >= 0.75:
        return "Very Happy"
    elif score >= 0.55:
        return "Happy"
    elif score >= 0.40:
        return "Neutral"
    else:
        return "Unhappy"
df['experience_level'] = df['experience_index'].apply(experience_label)

# Loyalty & Activity Logic

In [6]:
def loyalty_status(row):
    if row['Frequency'] >= df['Frequency'].quantile(0.75):
        return "Loyal"
    elif row['Frequency'] >= df['Frequency'].quantile(0.40):
        return "Active"
    else:
        return "Occasional"
df['loyalty_status'] = df.apply(loyalty_status, axis=1)

# Churn Risk Detection

In [7]:
def churn_risk(row):
    if row['Recency'] > df['Recency'].quantile(0.80):
        return "High Risk"
    elif row['Recency'] > df['Recency'].quantile(0.60):
        return "Medium Risk"
    else:
        return "Low Risk"
df['churn_risk'] = df.apply(churn_risk, axis=1)

# Customer Status

In [8]:
def final_status(row):
    if row['experience_level'] == "Very Happy" and row['loyalty_status'] == "Loyal":
        return "Very Happy & Loyal"
    if row['experience_level'] == "Happy" and row['loyalty_status'] in ["Loyal", "Active"]:
        return "Happy & Active"
    if row['churn_risk'] == "High Risk":
        return "Churn Risk"
    if row['churn_risk'] == "Medium Risk":
        return "At Risk"
    return "Regular Customer"
df['customer_status'] = df.apply(final_status, axis=1)

# Business Action Recommendation Engine

In [9]:
action_map = {
    "Very Happy & Loyal": "VIP rewards, early access to new products",
    "Happy & Active": "Cross-sell and personalized recommendations",
    "Regular Customer": "Discount nudges and engagement campaigns",
    "At Risk": "Re-engagement emails and limited-time offers",
    "Churn Risk": "High-value retention campaign with incentives"
}
df['recommended_action'] = df['customer_status'].map(action_map)

# Status Distribution

In [10]:
df['customer_status'].value_counts(normalize=True).round(3)

customer_status
Regular Customer    0.601
Churn Risk          0.200
At Risk             0.199
Happy & Active      0.000
Name: proportion, dtype: float64

# Save Output for App + LLM

In [11]:
df.to_csv("processed/customer_status_enriched.csv", index=False)